# Unigram Language model

# 1. Data Structures

In [1]:
from collections import defaultdict,Counter
from typing import Dict, List, Tuple
import math
import random

# 2. The MarkovChain Class


In [17]:
class MarkovChain:
  def __init__(self, order:int = 1):
    if order < 1:
      raise ValueError(f"Order must be >= 1, got {order}")
    self.order = order
    # counts[context_tuple][next_token] = Dict[Tuple[str, ...], Counter]
    self.counts : Dict[Tuple[str, ...], Counter] = defaultdict(Counter)
#    Output:  {
#   ('the', 'cat'): Counter({'sat': 2, 'ran': 1})
# }
    self.totals: Dict[Tuple[str, ...], int] = defaultdict(int)
    self.vocab: set = set()

  def train(self, tokens: List[str]) -> None:
    padded = ['<START>'] * self.order + tokens + ['<END>']
    for i in range(len(padded) - self.order):
      context = tuple(padded[i : i + self.order])
      next_token = padded[i+self.order]

      # update counts
      self.counts[context][next_token] += 1
      self.totals[context] += 1


      self.vocab.add(next_token)

  def generate(
      self,
      max_length: int = 100,
      temperature: float = 1.0,
      seed: str = ""
      ) -> str:

      """
          Generate text using ancestral/random sampling.

          Args:
              max_length: Maximum tokens to generate
              temperature: Sampling temperature (1.0 = unmodified)
              seed: Optional starting text

          Returns:
              Generated text as string
      """
      # Initialize context from seed or START tokens
      if seed:
          tokens = list(seed)
          #Use last 'order' tokens as context
          if len(tokens) >= self.order:
              context = tuple(tokens[-self.order:])
          else:
              padding = ['<START>'] * (self.order - len(tokens))
              context = tuple(padding + tokens)
          generated_tokens = list(seed)
      else:
          context = tuple(['<START>'] * self.order)
          generated_tokens = []

      # Generate tokens one at a time
      for _ in range(max_length):
          dist = self.get_distribution(context)
          if not dist:
              break
          if temperature != 1.0:
              dist = self._apply_temperature(dist, temperature)

          # Sample from distribution
          next_token = self._sample(dist)

          if next_token == '<END>':
              break

          generated_tokens.append(next_token)

          context = tuple(list(context)[1:] + [next_token])

      return ''.join(generated_tokens)

  def _apply_temperature(
      self,
      dist: Dict[str, float],
      temperature: float,

  )-> Dict[str, float]:
      """ Apply temperature scaling to distribution"""

      log_prob = {
          token: math.log(prob + 1e-10)
          for token, prob in dist.items()
      }


      exp_prob = {
          token: math.exp(lp / temperature)
          for token, lp in log_prob.items()
      }
      total = sum(exp_prob.values())


      if total == 0:

          return {token: 1.0 / len(exp_prob) for token in exp_prob.keys()} if exp_prob else {}
      else:
          return {
              token: prob / total
              for token, prob in exp_prob.items()
          }

  def _sample(self, dist: Dict[str, float]) -> str:
    tokens = list(dist.keys())
    probs = list(dist.values())

    if not tokens:
        return None

    if sum(probs) == 0:
        return random.choice(tokens)

    return random.choices(tokens, weights=probs, k = 1)[0]
  def probability(self, context: Tuple[str, ...], token:str) ->float:
    if context not in self.counts:
      return 0.0

    return self.counts[context][token] / self.totals[context]


  def get_distribution(self, context: Tuple[str, ...]) -> Dict[str, float]:
    """
          Get the full probability distribution P(* | context).

          Args:
              context: Tuple of previous tokens

          Returns:
              Dictionary mapping each possible next token to its probability.
              Empty dict if context never seen.
    """
    if context not in self.counts:
      return {}
    total = self.totals[context]
    return {
        token: count / total
        for token, count in self.counts[context].items()
    }
  def perplexity(self, tokens: List[str]) -> float:
    padded = ['<START>'] * self.order + tokens + ['<END>']

    log_prob_sum = 0.0
    n_tokens = 0

    for i in range(len(padded) - self.order):
      context = tuple(padded[i : i + self.order])
      next_token = padded[i + self.order]

      prob = self.probability(context, next_token)

      if prob == 0:
        return float('inf')

      log_prob_sum += math.log(prob)
      n_tokens += 1

    avg_neg_log_prob = -log_prob_sum / n_tokens

    return math.exp(avg_neg_log_prob)

#  3. Training

In [18]:
def train(self, tokens: List[str]) -> None:
        """
        Train the model by counting n-gram transitions.

        This implements Maximum Likelihood Estimation:
        P(token | context) = count(context, token) / count(context, *)

        Args:
            tokens: List of tokens (e.g., list of characters)
        """
        # Pad the sequence with START and END tokens
        # START tokens let us predict the first real tokens
        # END token lets the model learn when to stop
        padded = ['<START>'] * self.order + tokens + ['<END>']


        for i in range(len(padded) - self.order):

            context = tuple(padded[i : i + self.order])

            # Next token: the one right after the context
            next_token = padded[i + self.order]

            # Update counts
            self.counts[context][next_token] += 1
            self.totals[context] += 1


            self.vocab.add(next_token)

# 4. Probability Queries


# 5. Text Generation

In [19]:
def generate(
    self,
    max_length: int = 100,
    temperature: float = 1.0,
    seed: str = ""
    ) -> str:

    """
        Generate text using ancestral/random sampling.

        Args:
            max_length: Maximum tokens to generate
            temperature: Sampling temperature (1.0 = unmodified)
            seed: Optional starting text

        Returns:
            Generated text as string
    """
    # Initialize context from seed or START tokens
    if seed:
        tokens = list(seed)
        #Use last 'order' tokens as context
        if len(tokens) >= self.order:
            context = tuple(tokens[-self.order:])
        else:
            padding = ['<START>'] * (self.order - len(tokens))
            context = tuple(padding + tokens)
        generated_tokens = list(seed
    else:
        context = tuple(['<START>'] * self.order)
        generated_tokens = []

    # Generate tokens one at a time
    for _ in range(max_length):
        dist = self.get_distribution(context)
        if not dist:
            break
        if temperature != 1.0:
            dist = self._apply_temperature(dist, temperature)

        # Sample from distribution
        next_token = self._sample(dist)

        if next_token == '<END>':
            break

        generated_tokens.append(next_token)

        context = tuple(list(context)[1:] + [next_token])

    return ''.join(generated_tokens)

def _apply_temperature(
    self,
    dist: Dict[str, float],
    temperature: float,

)-> Dict[str, float]:
    """ Apply temperature scaling to distribution"""

    log_prob = {
        token: math.log(prob + 1e-10)
        for token, prob in dist.items()
    }

    # Re-evaluating the exp_prob calculation with temperature
    exp_prob = {
        token: math.exp(lp / temperature)
        for token, lp in log_prob.items()
    }
    total = sum(exp_prob.values())

    # Handle case where total might be zero if all log_prob become too small
    if total == 0:
        # Return a uniform distribution if probabilities become effectively zero after scaling
        return {token: 1.0 / len(exp_prob) for token in exp_prob.keys()} if exp_prob else {}
    else:
        return {
            token: prob / total
            for token, prob in exp_prob.items()
        }

def _sample(self, dist: Dict[str, float]) -> str:
  tokens = list(dist.keys())
  probs = list(dist.values())

  if not tokens:
      return None

  if sum(probs) == 0:
      return random.choice(tokens)

  return random.choices(tokens, weights=probs, k = 1)[0]

# Utility Methods

In [20]:
def num_states(self) -> int:
  return len(self.counts)

def __repr__(self) -> str:
  return f"MarkovChain(order={self.order}, states={self.num_states()}, vocab={len(self.vocab)})"


# Usage Example


In [21]:
text = """
To be, or not to be, that is the question:
Whether 'tis nobler in the mind to suffer
The slings and arrows of outrageous fortune,
Or to take arms against a sea of troubles
"""

tokens = list(text.lower())

model = MarkovChain(order=3)
model.train(tokens)


print(f"Model: {model}")


sample = model.generate(max_length=100, temperature=0.8)
print(f"Sample: {sample}")


train_ppl = model.perplexity(tokens)
print(f"Train perplexity: {train_ppl: 2f}")



Model: <__main__.MarkovChain object at 0x79c38f3b3d10>
Sample: 
to be, or to be, or to suffer
ther in that is the question:
whethe question:
whethe slings against 
Train perplexity:  1.157679
